calculate return period layer

In [ ]:
!apt-get update -qq
!apt-get install -y -qq gdal-bin libgdal-dev python3-gdal
!pip install -q rasterio

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3-numpy.
(Reading database ... 126284 files and directories currently installed.)
Preparing to unpack .../python3-numpy_1%3a1.21.5-1ubuntu22.04.1_amd64.deb ...
Unpacking python3-numpy (1:1.21.5-1ubuntu22.04.1) ...
Selecting previously unselected package python3-gdal.
Preparing to unpack .../python3-gdal_3.8.4+dfsg-1~jammy0_amd64.deb ...
Unpacking python3-gdal (3.8.4+dfsg-1~jammy0) ...
Selecting previously unselected package gdal-bin.
Preparing to unpack .../gdal-bin_3.8.4+dfsg-1~jammy0_amd64.deb ...
Unpacking gdal-bin (3.8.4+dfsg-1~jammy0) ...
Setting up python3-numpy (1:1.21.5-1ubuntu22.04.1) ...
Setting up python3-gdal (3.8.4+dfsg-1~jammy0) ...
Setting up gdal-bin (3.8.4+dfsg-1~jammy0) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━

In [ ]:
import os
import shutil
import numpy as np
import rasterio
from rasterio.windows import Window
from tqdm.contrib.concurrent import process_map
from functools import partial
from pathlib import Path
from osgeo import gdal


In [ ]:


# === Copy all .tif files to a new directory ===
def copy_tifs_to_directory(source_dir, tif_dir):
    tif_dir_path = Path(tif_dir)
    tif_dir_path.mkdir(parents=True, exist_ok=True)
    tif_files = list(Path(source_dir).glob("*.tif"))
    for tif_file in tif_files:
        shutil.copy(tif_file, tif_dir_path / tif_file.name)
    print(f"✅ Copied {len(tif_files)} .tif files to '{tif_dir}'")

# === Read land-sea mask file and create binary mask (1 = land, np.nan = sea/NaN) ===
def get_land_mask_from_file(land_mask_path):
    with rasterio.open(land_mask_path) as mask_src:
        mask_data = mask_src.read(1)
        land_mask = np.where(np.isnan(mask_data), np.nan, 1).astype(np.float32)
    print(f"✅ Loaded land-sea mask from: {land_mask_path}")
    return land_mask

def fit_empirical_tile_vectorized(data, mask, return_periods):
    years, h, w = data.shape
    flat = data.reshape(years, -1)  # [years, pixels]

    result = np.full((flat.shape[1], len(return_periods)), np.nan, dtype=np.float32)

    for px in range(flat.shape[1]):
        series = flat[:, px]
        series = series[~np.isnan(series)]
        n = len(series)
        if n >= 5:  # minimum years
            sorted_vals = np.sort(series)[::-1]  # descending
            ranks = np.arange(1, n + 1)
            emp_rps = (n + 1) / ranks  # Weibull plotting position
            for i, rp in enumerate(return_periods):
                if rp <= emp_rps[0]:
                    result[px, i] = np.interp(rp, emp_rps, sorted_vals)
                else:
                    # If requested return period > max in data, use max value (or np.nan)
                    result[px, i] = sorted_vals[0]

    # Reshape result back to raster grid
    result = result.reshape(h, w, len(return_periods))

    if mask is not None:
        for i in range(len(return_periods)):
            result[:, :, i] = np.where(np.isnan(mask), np.nan, result[:, :, i])

    return result



def process_tile(win, vrt_path, land_mask, return_periods):
    with rasterio.open(vrt_path) as src:
        data = src.read(window=win)
    mask = land_mask[win.row_off:win.row_off + win.height, win.col_off:win.col_off + win.width] if land_mask is not None else None
    result = fit_empirical_tile_vectorized(data, mask, return_periods)
    return (win, result)


def run_empirical_raster_pipeline(
    tif_dir, output_dir, output_basename, land_mask_path=None,
    return_periods=[10, 30, 50, 100]
):
    tif_paths = sorted([os.path.join(tif_dir, f) for f in os.listdir(tif_dir) if f.endswith('.tif')])
    vrt_path = os.path.join(tif_dir, "asis_stack.vrt")

    if not os.path.exists(vrt_path):
        print("🛠️ Building VRT stack virtually using GDAL...")
        gdal.BuildVRT(vrt_path, tif_paths, separate=True)
    else:
        print("✅ VRT stack already exists.")

    with rasterio.open(vrt_path) as src:
        height, width = src.height, src.width
        crs = src.crs
        transform = src.transform
        dtype = rasterio.float32

        meta = {
            'driver': 'GTiff',
            'height': height,
            'width': width,
            'count': 1,
            'dtype': dtype,
            'crs': crs,
            'transform': transform,
            'compress': 'lzw',
            'tiled': True,
            'blockxsize': 256,
            'blockysize': 256
        }

        all_windows = [
            Window(col, row, min(256, width - col), min(256, height - row))
            for row in range(0, height, 256)
            for col in range(0, width, 256)
        ]

    land_mask = get_land_mask_from_file(land_mask_path) if land_mask_path else None

    print(f"🚀 Processing {len(all_windows)} tiles with empirical return level estimation...")
    tile_func = partial(process_tile, vrt_path=vrt_path, land_mask=land_mask, return_periods=return_periods)
    results = process_map(tile_func, all_windows, max_workers=os.cpu_count(), chunksize=1, desc="Processing Tiles")

    os.makedirs(output_dir, exist_ok=True)
    for i, rp in enumerate(return_periods):
        out_path = os.path.join(output_dir, f"{output_basename}_{rp}yr.tif")
        with rasterio.open(out_path, 'w', **meta) as dst:
            for win, result in results:
                dst.write(result[:, :, i], 1, window=win)




In [ ]:
# Define the indicators
indicators = ['heatwave_duration', 'heatwave_frequency', 'heatwave_severity', 'extreme_heat_days']

# Root paths
source_root = "/content/drive/MyDrive/hwi_product/SIDS_updated_heatwave_indicators"
output_root = "/content/return_outputs"

# Loop through each indicator
for indicator in indicators:
    source_dir = f"{source_root}/{indicator}"
    output_basename = f"{indicator}_return_level"
    tif_dir = f"/content/{output_basename}"
    output_dir = f"{output_root}/{indicator}"
    land_mask_path = f"{source_dir}/1960.tif"  # Assumes 1960.tif is a valid mask for each

    print(f"\n=== Processing {indicator} ===")
    copy_tifs_to_directory(source_dir, tif_dir)
    run_empirical_raster_pipeline(tif_dir, output_dir, output_basename, land_mask_path)



=== Processing heatwave_duration ===
✅ Copied 64 .tif files to '/content/heatwave_duration_return_level'
🛠️ Building VRT stack virtually using GDAL...
✅ Loaded land-sea mask from: /content/drive/MyDrive/hwi_product/SIDS_updated_heatwave_indicators/heatwave_duration/1960.tif
🚀 Processing 90 tiles with empirical return level estimation...


/usr/local/lib/python3.11/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Processing Tiles:   0%|          | 0/90 [00:00<?, ?it/s]


=== Processing heatwave_frequency ===
✅ Copied 64 .tif files to '/content/heatwave_frequency_return_level'
🛠️ Building VRT stack virtually using GDAL...
✅ Loaded land-sea mask from: /content/drive/MyDrive/hwi_product/SIDS_updated_heatwave_indicators/heatwave_frequency/1960.tif
🚀 Processing 90 tiles with empirical return level estimation...


Processing Tiles:   0%|          | 0/90 [00:00<?, ?it/s]


=== Processing heatwave_severity ===
✅ Copied 64 .tif files to '/content/heatwave_severity_return_level'
🛠️ Building VRT stack virtually using GDAL...
✅ Loaded land-sea mask from: /content/drive/MyDrive/hwi_product/SIDS_updated_heatwave_indicators/heatwave_severity/1960.tif
🚀 Processing 90 tiles with empirical return level estimation...


Processing Tiles:   0%|          | 0/90 [00:00<?, ?it/s]


=== Processing extreme_heat_days ===
✅ Copied 64 .tif files to '/content/extreme_heat_days_return_level'
🛠️ Building VRT stack virtually using GDAL...
✅ Loaded land-sea mask from: /content/drive/MyDrive/hwi_product/SIDS_updated_heatwave_indicators/extreme_heat_days/1960.tif
🚀 Processing 90 tiles with empirical return level estimation...


Processing Tiles:   0%|          | 0/90 [00:00<?, ?it/s]

In [ ]:
import shutil
import os

# Parameters
indicators = ['heatwave_duration', 'heatwave_frequency', 'heatwave_severity', 'extreme_heat_days']
return_periods = [10, 30, 50, 100]
input_root = "/content/return_outputs"
drive_folder = "/content/drive/MyDrive/CCRI/return_levels"

# Ensure output directory exists
os.makedirs(drive_folder, exist_ok=True)

# Loop through each indicator and return period
for indicator in indicators:
    output_basename = f"{indicator}_return_level"
    for rp in return_periods:
        src_path = os.path.join(input_root, indicator, f"{output_basename}_{rp}yr.tif")
        dst_path = os.path.join(drive_folder, f"{output_basename}_{rp}yr.tif")
        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
            print(f"✅ Copied {src_path} to {dst_path}")
        else:
            print(f"❌ Missing file: {src_path}")


✅ Copied /content/return_outputs/heatwave_duration/heatwave_duration_return_level_10yr.tif to /content/drive/MyDrive/CCRI/return_levels/heatwave_duration_return_level_10yr.tif
✅ Copied /content/return_outputs/heatwave_duration/heatwave_duration_return_level_30yr.tif to /content/drive/MyDrive/CCRI/return_levels/heatwave_duration_return_level_30yr.tif
✅ Copied /content/return_outputs/heatwave_duration/heatwave_duration_return_level_50yr.tif to /content/drive/MyDrive/CCRI/return_levels/heatwave_duration_return_level_50yr.tif
✅ Copied /content/return_outputs/heatwave_duration/heatwave_duration_return_level_100yr.tif to /content/drive/MyDrive/CCRI/return_levels/heatwave_duration_return_level_100yr.tif
✅ Copied /content/return_outputs/heatwave_frequency/heatwave_frequency_return_level_10yr.tif to /content/drive/MyDrive/CCRI/return_levels/heatwave_frequency_return_level_10yr.tif
✅ Copied /content/return_outputs/heatwave_frequency/heatwave_frequency_return_level_30yr.tif to /content/drive/MyDr